# SME Liquidity GRPO Training on Colab

This notebook follows the same simple package-first pattern as `kube_sre_gym_colab.ipynb`, but for the SME liquidity environment.

| Component | Detail |
|-----------|--------|
| Environment | In-process SME liquidity environment from this repo |
| Training | Simple GRPO notebook wrapper |
| Model | `Qwen/Qwen3-0.6B` + LoRA |
| Framework | HF TRL GRPO using the canonical repo training path |

## 1. Install Dependencies

Install the RL dependencies needed for the canonical liquidity trainer.

In [ ]:
%pip install -q "trl==0.29.0" "transformers>=4.56.0" "accelerate>=1.0.0" "peft>=0.17.0" bitsandbytes sentencepiece matplotlib datasets

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git"
REPO_BRANCH = "round1_baseline"
REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")

def ensure_liquidity_repo() -> Path:
    cwd = Path.cwd()
    if (cwd / "rl" / "train_grpo_liquidity.py").exists():
        return cwd

    if not REPO_DIR.exists():
        subprocess.check_call(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", "-q", REPO_URL, str(REPO_DIR)]
        )
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

    os.chdir(REPO_DIR)
    repo_root = Path.cwd()
    module_path = repo_root / "rl" / "train_grpo_liquidity.py"
    if not module_path.exists():
        raise FileNotFoundError(
            f"Expected module not found at {module_path}. Check that branch {REPO_BRANCH!r} contains the file."
        )
    return repo_root

repo_root = ensure_liquidity_repo()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[rl]"])

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Using repo root: {repo_root}")
print(f"Using repo branch: {REPO_BRANCH}")


## 2. Configuration

Set the model, environment task, and small profile. This notebook defaults to a T4-safe `tiny` profile.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")

RUN_PROFILE = "tiny"
MODEL_NAME = "Qwen/Qwen3-0.6B"
TASK_NAME = "liquidity-correlation-hard"
DIFFICULTY = "hard"
TOTAL_PERIODS = 2
SEED_BASE = 1000
USE_VLLM = False
OUTPUT_DIR = Path("outputs/grpo_sme_liquidity_simple")

print(f"Profile     : {RUN_PROFILE}")
print(f"Model       : {MODEL_NAME}")
print(f"Task        : {TASK_NAME}")
print(f"Difficulty  : {DIFFICULTY}")
print(f"Periods     : {TOTAL_PERIODS}")
print(f"Seed base   : {SEED_BASE}")
print(f"Use vLLM    : {USE_VLLM}")
print(f"Output dir  : {OUTPUT_DIR}")
print("Note        : 'tiny' is a smoke-sized run and will produce a sparse reward curve.")
print("Recommendation: switch to profile='small' or raise num_samples/max_steps for a more meaningful curve.")


## 3. Import Training Utilities from Package

All notebook logic comes from the repo package. The notebook stays thin and reuses the same Python training functions as the CLI script.

In [ ]:
try:
    from rl.train_grpo_liquidity import (
        build_canonical_training_args,
        build_run_plan,
        build_training_session,
        make_training_args,
        plot_rewards,
        run_training,
        smoke_test_environment,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import rl.train_grpo_liquidity. Run the install/bootstrap cell first so the notebook checks out "
        "the correct repo branch and installs the package in editable mode."
    ) from exc

NOTEBOOK_ARGS = make_training_args(
    profile=RUN_PROFILE,
    model_name=MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
    seed_base=SEED_BASE,
    output_dir=str(OUTPUT_DIR),
    use_vllm=USE_VLLM,
)
CANONICAL_ARGS = build_canonical_training_args(NOTEBOOK_ARGS)
RUN_PLAN = build_run_plan(NOTEBOOK_ARGS, CANONICAL_ARGS)

RUN_PLAN


## 4. Smoke Test

Run a cheap reset-only smoke test first so the environment wiring is checked before training.

In [ ]:
smoke = smoke_test_environment(CANONICAL_ARGS)
print("Smoke test passed. Environment is ready for training.")
smoke


## 5. Train!

Launch the simple canonical GRPO training run. This uses the same Python entrypoint as `python -m rl.train_grpo_liquidity`.

In [ ]:
print("Starting GRPO training...")
print(f"  Profile     : {RUN_PROFILE}")
print(f"  Model       : {MODEL_NAME}")
print(f"  Task        : {TASK_NAME}")
print(f"  Difficulty  : {DIFFICULTY}")
print(f"  Output dir  : {OUTPUT_DIR}")
print()

print(f"Resolved canonical output dir: {RUN_PLAN['paths']['output_dir']}")

manifest = run_training(NOTEBOOK_ARGS)
print(f"Checkpoint path: {manifest['training']['checkpoint_path']}")
print(f"Episode reward log path: {manifest['training']['episode_reward_log_path']}")
print(f"Reward curve path: {manifest['training']['reward_curve_path']}")
manifest["training"]


## 6. Reward Curves

Visualize training progress using `plot_rewards` from the package.

In [ ]:
from pathlib import Path

episode_reward_log_path = Path(manifest["training"]["episode_reward_log_path"])
output_dir = Path(manifest["plan"]["paths"]["output_dir"])

# Use plot_rewards from the package
plot_rewards(episode_reward_log_path, output_dir / "reward_curve.png")

# Also display inline
import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(str(output_dir / "reward_curve.png"))
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

print(f"Episode reward log saved to {episode_reward_log_path}")
print(f"Reward curve saved to {output_dir / 'reward_curve.png'}")


## 7. Exported Artifacts

The simple script writes a run manifest plus the main training artifacts.

In [ ]:
print(f"Checkpoint path: {manifest['training']['checkpoint_path']}")
print(f"Episode reward log path: {manifest['training']['episode_reward_log_path']}")
print(f"Trainer reward log path: {manifest['training']['trainer_reward_log_path']}")
print(f"Reward curve path: {manifest['training']['reward_curve_path']}")
print(f"Training dashboard path: {manifest['training']['training_dashboard_path']}")
print(f"Run manifest path: {manifest['manifest_path']}")

manifest
